## Data Pre-processing

### 0) Setup

In [ ]:
import pandas as pd
import numpy as np

# 0) Start from your pipeline output
from scripts.preprocess_data import process_data
df = process_data().copy()

# 1) Drop aggregates (keep sovereigns)
AGG = {"WLD","EUU","HIC","UMC","LMC","LIC","LMY","SSA","ECS","EAS","AMR","AFR","SEAR","WPR","EUR","EMR","MEA","OED","NAC","LAC","EAP","SAS","TEC","PRE","PST","OSS","CSS","SST","TLA","TSA","TMN","TEA"}
df = df[~df["Country"].isin(AGG)].copy()

# 2) Keep common, recent window
df = df[(df["Year"] >= 2010) & (df["Year"] <= 2021)].copy()

# 3) Resolve duplicates within (Indicator, Country, Year)
# Rule: if both PAA and PAC exist for same country-year, we will keep PAA downstream.
# For exact duplicates inside the same indicator → average.
dup_counts = df.groupby(["Indicator","Country","Year"]).size()
if (dup_counts > 1).any():
    df = (df.groupby(["Indicator","Country","Year"], as_index=False)["Value"]
            .mean())

# 4) Optional: light interpolation within each (Country, Indicator)
def _interp(g):
    g = g.sort_values("Year").copy()
    g["Value"] = g["Value"].interpolate(limit=2)  # allow small gaps
    g["Value"] = g["Value"].bfill(limit=1).ffill(limit=1)
    return g

df = df.groupby(["Country","Indicator"], group_keys=False).apply(_interp)

# 5) Pivot to wide panel
keep = ["NY.GDP.PCAP.CD","SH.XPD.CHEX.PC.CD","NCD_PAC","NCD_PAA","SA_0000001400","NCD_UNDER70","WHS2_131"]
panel = (df[df["Indicator"].isin(keep)]
         .pivot_table(index=["Country","Year"], columns="Indicator", values="Value"))

# Prefer age-standardized inactivity for modeling: create one column INACTIVITY
panel["INACTIVITY"] = panel["NCD_PAA"].combine_first(panel["NCD_PAC"])

# Require target + key contexts present
panel = panel.dropna(subset=["WHS2_131","NY.GDP.PCAP.CD","SH.XPD.CHEX.PC.CD","INACTIVITY"])

# 6) Helper to compute last-3yr level & slope (per country, per indicator)
def level_and_slope(s):
    s = s.dropna()
    if s.empty:
        return pd.Series({"level": np.nan, "slope": np.nan})
    # level = mean of last up to 3 values
    level = s.sort_index().tail(3).mean()
    # slope via OLS on Year (simple polyfit)
    x = s.index.values.astype(float)
    y = s.values.astype(float)
    slope = np.polyfit(x, y, 1)[0] if len(s) >= 2 else np.nan
    return pd.Series({"level": level, "slope": slope})

# 7) Build country-level features
indicators_for_feats = {
    "INACTIVITY": "INACTIVITY",
    "ALCOHOL": "SA_0000001400",
    "NCD_MORT": "WHS2_131",
    "PREMATURE": "NCD_UNDER70",
    "GDP_PC": "NY.GDP.PCAP.CD",
    "HEALTH_PC": "SH.XPD.CHEX.PC.CD",
}

feat_frames = []
for nice, col in indicators_for_feats.items():
    if col not in panel.columns:
        continue
    tmp = (panel[col]
           .unstack(0)                 # Year × Country
           .apply(level_and_slope))    # returns 2 rows (level/slope) per Country
    # After apply, transpose to Country × {level,slope}
    tmp = tmp.T
    tmp.columns = [f"{nice}_{c}" for c in tmp.columns]
    feat_frames.append(tmp)

features = pd.concat(feat_frames, axis=1)

# 8) Final tidy artifacts
# a) Yearly panel for downstream plotting
panel_out = panel.reset_index()
# b) Country-level feature matrix for clustering/regression
features_out = features.sort_index()

# Save (optional)
panel_out.to_parquet("artifacts\\panel_2010_2021.parquet", index=False)
features_out.to_parquet("artifacts/features_country_level.parquet")
print(panel_out.shape, features_out.shape)


Loading World Bank data...
  > gdp_per_capita_data - Done
  > health_per_capita_data - Done
Loading WHO data...
  > NCD_PAC - Done
  > NCD_PAA - Done
  > alcohol - Done
  > NCD_UNDER70 - Done
  > WHS2_131 - Done
Dataframes processed: ['gdp_per_capita_data', 'health_per_capita_data', 'NCD_PAC', 'NCD_PAA', 'alcohol', 'NCD_UNDER70', 'WHS2_131']
Merged dataframe:
  # of rows:    139318
  # of columns: 4
Cleaning info:
  Original # of rows:      139318
  Duplicate rows dropped:  3119
  Na values dropped:       14574
  New # of rows:           121625
Unique Indicators:
['NY.GDP.PCAP.CD' 'SH.XPD.CHEX.PC.CD' 'NCD_PAC' 'NCD_PAA' 'SA_0000001400'
 'NCD_UNDER70' 'WHS2_131']

Unique Countries:
['AFE' 'AFW' 'ARG' 'AUS' 'AUT' 'BDI' 'BEL' 'BEN' 'BFA' 'BGD' 'BHS' 'BLZ'
 'BMU' 'BOL' 'BRA' 'BRB' 'BWA' 'CAF' 'CAN' 'CHE' 'CHL' 'CHN' 'CIV' 'CMR'
 'COD' 'COG' 'COL' 'CRI' 'CSS' 'DEU' 'DNK' 'DOM' 'DZA' 'EAP' 'EAR' 'EAS'
 'ECS' 'ECU' 'EGY' 'EMU' 'ESP' 'ETH' 'EUU' 'FCS' 'FIN' 'FJI' 'FRA' 'GAB'
 'GBR' 'GHA' 'GRC'

C:\Users\sla99\AppData\Local\Temp\ipykernel_9784\4051242088.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(["Country","Indicator"], group_keys=False).apply(_interp)


(2157, 10) (181, 12)


In [ ]:
from scripts.preprocess_data import process_data
import pandas as pd, numpy as np
PATH = "artifacts"

df = process_data().copy()  # merged long, cleaned
df.to_parquet(f"{PATH}/merged_long.parquet", index=False)

# drop aggregates, 2010–2021, dedupe
AGG = {"WLD","EUU","HIC","UMC","LMC","LIC","LMY","SSA","ECS","EAS","AMR","AFR","SEAR","WPR","EUR","EMR","MEA","OED","NAC","LAC","EAP","SAS","TEC","PRE","PST","OSS","CSS","SST","TLA","TSA","TMN","TEA"}
df = df[~df.Country.isin(AGG) & df.Year.between(2010, 2021)]

df = df.groupby(["Indicator","Country","Year"], as_index=False)["Value"].mean()
keep = ["NY.GDP.PCAP.CD","SH.XPD.CHEX.PC.CD","NCD_PAC","NCD_PAA","SA_0000001400","NCD_UNDER70","WHS2_131"]
p = df[df.Indicator.isin(keep)].pivot_table(index=["Country","Year"], columns="Indicator", values="Value")
p["INACTIVITY"] = p["NCD_PAA"].combine_first(p["NCD_PAC"])
p.reset_index().to_parquet(f"{PATH}/panel_wide_2010_2021.parquet", index=False)

panel_reg = p.dropna(subset=["WHS2_131","NY.GDP.PCAP.CD","SH.XPD.CHEX.PC.CD"]).reset_index()
panel_reg.to_parquet(f"{PATH}/panel_reg.parquet", index=False)

# country features (Level & Slope)
def level_and_slope(s):
    s = s.dropna().sort_index()
    level = s.tail(3).mean() if len(s) else np.nan
    slope = np.polyfit(s.index.values, s.values, 1)[0] if len(s) >= 2 else np.nan
    return pd.Series({"level": level, "slope": slope})

ind_map = {
    "INACTIVITY": p["INACTIVITY"], "ALCOHOL": p["SA_0000001400"], "NCD_MORT": p["WHS2_131"],
    "PREMATURE": p["NCD_UNDER70"], "GDP_PC": p["NY.GDP.PCAP.CD"], "HEALTH_PC": p["SH.XPD.CHEX.PC.CD"],
}
feat_parts = []
for name, col in ind_map.items():
    tmp = col.unstack(0).apply(level_and_slope)  # Year×Country -> features
    tmp = tmp.T; tmp.columns = [f"{name}_{c}" for c in tmp.columns]; feat_parts.append(tmp)
features = pd.concat(feat_parts, axis=1).sort_index()
features.to_parquet(f"{PATH}/features_country_level.parquet")


Loading World Bank data...
  > gdp_per_capita_data - Done
  > health_per_capita_data - Done
Loading WHO data...
  > NCD_PAC - Done
  > NCD_PAA - Done
  > alcohol - Done
  > NCD_UNDER70 - Done
  > WHS2_131 - Done
Dataframes processed: ['gdp_per_capita_data', 'health_per_capita_data', 'NCD_PAC', 'NCD_PAA', 'alcohol', 'NCD_UNDER70', 'WHS2_131']
Merged dataframe:
  # of rows:    139318
  # of columns: 4
Cleaning info:
  Original # of rows:      139318
  Duplicate rows dropped:  3119
  Na values dropped:       14574
  New # of rows:           121625
Unique Indicators:
['NY.GDP.PCAP.CD' 'SH.XPD.CHEX.PC.CD' 'NCD_PAC' 'NCD_PAA' 'SA_0000001400'
 'NCD_UNDER70' 'WHS2_131']

Unique Countries:
['AFE' 'AFW' 'ARG' 'AUS' 'AUT' 'BDI' 'BEL' 'BEN' 'BFA' 'BGD' 'BHS' 'BLZ'
 'BMU' 'BOL' 'BRA' 'BRB' 'BWA' 'CAF' 'CAN' 'CHE' 'CHL' 'CHN' 'CIV' 'CMR'
 'COD' 'COG' 'COL' 'CRI' 'CSS' 'DEU' 'DNK' 'DOM' 'DZA' 'EAP' 'EAR' 'EAS'
 'ECS' 'ECU' 'EGY' 'EMU' 'ESP' 'ETH' 'EUU' 'FCS' 'FIN' 'FJI' 'FRA' 'GAB'
 'GBR' 'GHA' 'GRC'

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# --- 0) Expect panel_reg with columns: Country, Year, WHS2_131, NY.GDP.PCAP.CD, SH.XPD.CHEX.PC.CD
assert {"Country","Year","WHS2_131","NY.GDP.PCAP.CD","SH.XPD.CHEX.PC.CD"}.issubset(panel_reg.columns)

dfA = panel_reg.copy()

# --- 1) (Recommended) log-transform skewed money vars; mortality stays in level
dfA["ln_gdp_pc"] = np.log1p(dfA["NY.GDP.PCAP.CD"])
dfA["ln_health_pc"] = np.log1p(dfA["SH.XPD.CHEX.PC.CD"])

# --- 2) Fit OLS with robust (HC1) SEs
y = dfA["WHS2_131"].astype(float)
X = sm.add_constant(dfA[["ln_gdp_pc","ln_health_pc"]])
modelA = sm.OLS(y, X).fit(cov_type="HC1")
print(modelA.summary())

# --- 3) Predictions + residuals
dfA["yhat"] = modelA.predict(X)
dfA["residual"] = y - dfA["yhat"]           # < 0 = better than expected; > 0 = worse

# --- 4) Leaderboards
# per year (who’s best/worst this year)
yearly_top = (dfA.sort_values(["Year","residual"])
                .groupby("Year")
                .head(10)[["Year","Country","WHS2_131","yhat","residual"]])

yearly_bottom = (dfA.sort_values(["Year","residual"], ascending=[True, False])
                   .groupby("Year")
                   .head(10)[["Year","Country","WHS2_131","yhat","residual"]])

# across years (consistent over/under-performers)
country_summary = (dfA.groupby("Country")["residual"]
                     .agg(["count","mean","median"])
                     .rename(columns={"mean":"resid_mean","median":"resid_median"})
                     .sort_values("resid_mean"))

print("Best (most negative residuals):")
print(country_summary.head(15))
print("\nWorst (most positive residuals):")
print(country_summary.tail(15))

# --- 5) (Optional) Save artifacts for the report
dfA.to_parquet("artifacts/modelA_predictions.parquet", index=False)
country_summary.to_csv("artifacts/modelA_country_residuals.csv")


                            OLS Regression Results                            
Dep. Variable:               WHS2_131   R-squared:                       0.419
Model:                            OLS   Adj. R-squared:                  0.419
Method:                 Least Squares   F-statistic:                     1257.
Date:                Sat, 11 Oct 2025   Prob (F-statistic):               0.00
Time:                        12:40:42   Log-Likelihood:                -13629.
No. Observations:                2157   AIC:                         2.726e+04
Df Residuals:                    2154   BIC:                         2.728e+04
Df Model:                           2                                         
Covariance Type:                  HC1                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const         1119.8197     37.578     29.800   

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

d = panel_reg.copy()

# Numeric features
d["ln_gdp_pc"] = np.log1p(d["NY.GDP.PCAP.CD"])
d["ln_health_pc"] = np.log1p(d["SH.XPD.CHEX.PC.CD"])

# 1) Clean y and numeric X
y = d["WHS2_131"].astype(float)
X_num = d[["ln_gdp_pc","ln_health_pc"]].astype(float)

# 2) Year fixed effects as float dummies (built separately)
X_year = pd.get_dummies(d["Year"].astype("category"), drop_first=True, dtype=float)

# 3) Combine and add constant; ensure all float and aligned
X = pd.concat([X_num, X_year], axis=1).astype(float)
X = sm.add_constant(X)

# 4) Drop any rows with NaNs in y/X (just in case)
mask = ~pd.isna(y) & ~X.isna().any(axis=1)
y2 = y[mask]
X2 = X.loc[mask]

# 5) OLS with clustered SEs by country
m_fe = sm.OLS(y2, X2).fit(cov_type="cluster", cov_kwds={"groups": d.loc[mask, "Country"]})
print(m_fe.summary())

# Predictions + residuals
d.loc[mask, "yhat_fe"] = m_fe.predict(X2)
d["resid_fe"] = d["WHS2_131"] - d["yhat_fe"]

                            OLS Regression Results                            
Dep. Variable:               WHS2_131   R-squared:                       0.423
Model:                            OLS   Adj. R-squared:                  0.420
Method:                 Least Squares   F-statistic:                     26.65
Date:                Sat, 11 Oct 2025   Prob (F-statistic):           2.75e-35
Time:                        12:42:49   Log-Likelihood:                -13622.
No. Observations:                2157   AIC:                         2.727e+04
Df Residuals:                    2143   BIC:                         2.735e+04
Df Model:                          13                                         
Covariance Type:              cluster                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const         1145.4340    120.135      9.535   

---

### Impute + scale + cluster in a pipeline

In [14]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score

feats = pd.read_parquet("artifacts/features_country_level.parquet")

# 1) keep only numeric feature columns
X = feats.select_dtypes(include=[np.number]).copy()

# 2) (optional) drop columns that are almost empty or constant
too_sparse = X.isna().mean() > 0.5
const = X.nunique(dropna=True) <= 1
X = X.loc[:, ~(too_sparse | const)]

# 3) build pipeline: median impute -> z-score -> KMeans
pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("kmeans", KMeans(n_clusters=4, n_init=50, random_state=42)),
])

labels = pipe.fit_predict(X)
feats["cluster"] = labels

# Quality check
X_z = pipe.named_steps["scale"].transform(pipe.named_steps["impute"].transform(X))
print("Silhouette:", round(silhouette_score(X_z, labels), 3))
print(feats["cluster"].value_counts().sort_index())

# Save
feats.to_parquet("artifacts/features_country_level_clustered.parquet")


c:\Users\sla99\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


Silhouette: 0.141
cluster
0    98
1    28
2    53
3    65
Name: count, dtype: int64


### Pick k systematically: try k=3–8 and choose the best silhouette (or the simplest k within ±0.01 of the best).

In [15]:
# 1) choose k
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np, pandas as pd, os

os.environ["OMP_NUM_THREADS"] = "1"  # fix MKL leak warning on Windows

feats = pd.read_parquet("artifacts/features_country_level.parquet")
X = feats.select_dtypes("number").copy()

# optional: prune near-constant / too-sparse
mask = (X.isna().mean() <= 0.5) & (X.nunique(dropna=True) > 2)
X = X.loc[:, mask]

pipe = Pipeline([("imp", SimpleImputer(strategy="median")),
                 ("sc", StandardScaler())])

Z = pipe.fit_transform(X)

best = {}
for k in range(3, 9):
    km = KMeans(n_clusters=k, n_init=50, random_state=42)
    labels = km.fit_predict(Z)
    s = silhouette_score(Z, labels)
    best[k] = (s, labels, km)
k_best = max(best, key=lambda k: best[k][0])
s_best, labels, km = best[k_best]
feats["cluster"] = labels
print("Best k:", k_best, "Silhouette:", round(s_best, 3))

# 2) quick cluster profile (means in original units)
prof = (pd.DataFrame(pipe.named_steps["imp"].transform(feats.select_dtypes("number").drop(columns=["cluster"], errors="ignore")),
                      index=feats.index,
                      columns=X.columns)
        .join(feats["cluster"])
        .groupby("cluster").mean().round(2))
print(prof)

feats.to_parquet("artifacts/features_country_level_clustered.parquet")

c:\Users\sla99\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\sla99\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\sla99\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\sla99\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Window

Best k: 3 Silhouette: 0.182
         INACTIVITY_level  INACTIVITY_slope  ALCOHOL_level  ALCOHOL_slope  \
cluster                                                                     
0                   21.92             -0.06           1.38          -0.01   
1                   22.17             -0.34           3.48          -0.02   
2                   37.38              0.46           1.90          -0.01   

         NCD_MORT_level  NCD_MORT_slope  PREMATURE_level  PREMATURE_slope  \
cluster                                                                     
0                627.02           -4.80            54.72            -0.01   
1                351.88           -5.26            25.34            -0.27   
2                482.88           -2.72            40.41            -0.28   

         GDP_PC_level  GDP_PC_slope  HEALTH_PC_level  HEALTH_PC_slope  
cluster                                                                
0            10326.17         95.79           324.43    

In [16]:
# assumes you already computed FE residuals into panel_reg-like df `d` with columns Country, Year, resid_fe
d = pd.read_parquet("artifacts/modelA_predictions.parquet") if False else d  # use your existing df
clusters = pd.read_parquet("artifacts/features_country_level_clustered.parquet")["cluster"]
rank_in_cluster = (d.groupby("Country")["resid_fe"].mean()
                     .to_frame("resid_mean_fe").join(clusters)
                     .dropna(subset=["cluster"])
                     .sort_values(["cluster","resid_mean_fe"]))

# best/worst per cluster (top/bottom 5)
out = (rank_in_cluster.groupby("cluster")
        .apply(lambda g: pd.concat([g.head(5), g.tail(5)])))
print(out)


                 resid_mean_fe  cluster
cluster Country                        
0       ETH        -260.572293        0
        TZA        -218.836340        0
        BGD        -203.235678        0
        NER        -173.599017        0
        NPL        -169.541977        0
        LSO         334.627916        0
        SWZ         386.992355        0
        FJI         399.054741        0
        FSM         406.348711        0
        KIR         445.020559        0
1       SGP        -146.010613        1
        ESP        -113.537328        1
        ISR        -105.593555        1
        MLT         -96.530169        1
        FRA         -85.922479        1
        DNK          10.545311        1
        EST          37.171239        1
        USA          60.036704        1
        LTU         109.939159        1
        LVA         128.231029        1
2       PER        -268.257615        2
        NIC        -248.952544        2
        THA        -193.426542        2


C:\Users\sla99\AppData\Local\Temp\ipykernel_9784\3165609466.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.concat([g.head(5), g.tail(5)])))


### 0) Setup (fix warnings + load artifacts)

In [17]:
import os, pandas as pd, numpy as np
os.environ["OMP_NUM_THREADS"] = "1"  # suppress MKL KMeans warning on Windows

# Inputs produced earlier
feats = pd.read_parquet("artifacts/features_country_level.parquet")       # 1 row / country, level & slope features
panel = pd.read_parquet("artifacts/panel_2010_2021.parquet")              # Country–Year wide panel
modelA = pd.read_parquet("artifacts/modelA_predictions.parquet")          # has yhat/residual; or use your FE df `d`
# If you computed FE residuals in `d`, do: modelA = d.copy()

### 1) Finalize clustering at k=3 and profile clusters

In [18]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline

X = feats.select_dtypes("number").copy()
mask = (X.isna().mean() <= 0.5) & (X.nunique(dropna=True) > 2)
X = X.loc[:, mask]

pipe = Pipeline([("imp", SimpleImputer(strategy="median")),
                 ("sc", StandardScaler())])
Z = pipe.fit_transform(X)

kmeans = KMeans(n_clusters=3, n_init=50, random_state=42)
labels = kmeans.fit_predict(Z)
feats["cluster"] = labels

# Cluster profile in original units (means)
X_imp = pd.DataFrame(pipe.named_steps["imp"].transform(X), index=feats.index, columns=X.columns)
profile = X_imp.join(feats["cluster"]).groupby("cluster").mean().round(2)
profile.to_csv("artifacts/cluster_profile.csv")
feats[["cluster"]].to_csv("artifacts/country_clusters.csv")
print(profile)

c:\Users\sla99\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


         INACTIVITY_level  INACTIVITY_slope  ALCOHOL_level  ALCOHOL_slope  \
cluster                                                                     
0                   21.92             -0.06           1.38          -0.01   
1                   22.17             -0.34           3.48          -0.02   
2                   37.38              0.46           1.90          -0.01   

         NCD_MORT_level  NCD_MORT_slope  PREMATURE_level  PREMATURE_slope  \
cluster                                                                     
0                627.02           -4.80            54.72            -0.01   
1                351.88           -5.26            25.34            -0.27   
2                482.88           -2.72            40.41            -0.28   

         GDP_PC_level  GDP_PC_slope  HEALTH_PC_level  HEALTH_PC_slope  
cluster                                                                
0            10326.17         95.79           324.43             5.48  
1           

### 2) “Better / worse than expected” within each cluster
Use FE residuals if created, else baseline residuals. Negative = better (lower mortality than predicted).

In [19]:
# Join residuals with clusters (one row per Country–Year in modelA)
resid_col = "resid_fe" if "resid_fe" in modelA.columns else "residual"
country_resid = (modelA.groupby("Country")[resid_col]
                 .mean().to_frame("resid_mean")).join(feats["cluster"]).dropna()

# Leaderboard: top/bottom 5 per cluster
lb = (country_resid.sort_values(["cluster","resid_mean"])
        .groupby("cluster")
        .apply(lambda g: pd.concat([g.head(5), g.tail(5)]))
        .reset_index(level=0, drop=True))
lb.to_csv("artifacts/cluster_residual_leaderboard.csv")
print(lb.head(20))


         resid_mean  cluster
Country                     
ETH     -261.479248        0
TZA     -219.593416        0
BGD     -205.043430        0
NER     -173.757336        0
NPL     -169.875497        0
LSO      336.098961        0
SWZ      387.616753        0
FJI      398.151874        0
FSM      408.177000        0
KIR      446.171732        0
SGP     -146.902261        1
ESP     -112.520402        1
ISR     -105.082058        1
MLT      -95.651781        1
FRA      -84.415912        1
DNK       11.812493        1
EST       37.413886        1
USA       62.335250        1
LTU      110.222758        1
LVA      128.416383        1


C:\Users\sla99\AppData\Local\Temp\ipykernel_9784\2997378188.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.concat([g.head(5), g.tail(5)]))


### 3) “Unusual cases” (outliers) by distance to centroid in feature space

In [20]:
# Distance to assigned centroid
centers = kmeans.cluster_centers_
dists = np.linalg.norm(Z - centers[labels], axis=1)
feats["dist_to_center"] = dists

outliers = (feats[["cluster","dist_to_center"]]
            .sort_values(["cluster","dist_to_center"], ascending=[True, False])
            .groupby("cluster").head(8))  # top 8 farthest per cluster
outliers.to_csv("artifacts/cluster_outliers.csv")
print(outliers.head(20))

         cluster  dist_to_center
Country                         
BFA            0        7.607991
UZB            0        5.712030
AZE            0        5.613130
MNG            0        4.341375
BMU            0        4.333632
BLR            0        4.264936
LSO            0        4.209037
RUS            0        4.169499
USA            1        8.054325
MCO            1        7.488122
LIE            1        6.890653
IRL            1        6.066963
NOR            1        5.835960
ISL            1        5.615965
EST            1        5.499923
LVA            1        4.412847
QAT            2        7.542981
MDV            2        7.074920
KOR            2        5.287717
VEN            2        5.272872


### 4) Map: choropleth of clusters + Time-series: cluster median mortality trend

In [21]:
import plotly.express as px

# Map (needs ISO3 codes in Country column; your codes are ISO3-like)
map_df = feats[["cluster"]].reset_index().rename(columns={"index":"Country"})
fig_map = px.choropleth(map_df, locations="Country", color="cluster",
                        color_continuous_scale="Viridis", locationmode="ISO-3",
                        title="Country clusters (k=3)")
fig_map.write_html("artifacts/map_clusters.html")

# Cluster median mortality trend over time
cols_needed = ["Country","Year","WHS2_131"]
mort = panel[cols_needed].dropna()
mort = mort.merge(feats[["cluster"]], left_on="Country", right_index=True, how="left")
trend = (mort.groupby(["cluster","Year"])["WHS2_131"].median().reset_index())
fig_ts = px.line(trend, x="Year", y="WHS2_131", color="cluster",
                 title="Median NCD mortality (per 100k) by cluster")
fig_ts.write_html("artifacts/cluster_mortality_trends.html")

In [27]:
# Prep
import os, pandas as pd, numpy as np
import plotly.express as px
os.environ["OMP_NUM_THREADS"] = "1"  # quiet MKL warning

feats = pd.read_parquet("artifacts/features_country_level.parquet")      # country features
clusters = pd.read_parquet("artifacts/country_clusters.parquet") if os.path.exists("artifacts/country_clusters.parquet") else None
if clusters is None:
    # if you saved clusters as CSV:
    clusters = pd.read_csv("artifacts/country_clusters.csv", index_col=0)
feats = feats.join(clusters)  # adds 'cluster' column
feats = feats.reset_index().rename(columns={"index":"Country"})  # ensure Country column exists

### 5) Small “over/under-performance” summary table

In [34]:
# country_resid: index=Country, columns=['resid_mean','cluster']
# ctx: per-country medians for GDP/Health (as you built earlier)

# 1) Rank within each cluster (no apply, no duplicate levels)
cr = country_resid.join(ctx, how="left").copy()
cr["rank_in_cluster"] = cr.groupby("cluster")["resid_mean"].rank(method="dense")

# 2) Order: better (more negative) first within each cluster
summ = cr.sort_values(["cluster", "resid_mean"]).reset_index()  # brings Country out of index

# 3) Save + peek
summ.to_csv("artifacts/over_under_summary.csv", index=False)
print(summ.head(15))


   Country  resid_mean  cluster   gdp_pc_med  health_pc_med  rank_in_cluster
0      ETH -261.479248        0   659.020879      23.369365              1.0
1      TZA -219.593416        0   974.865570      38.393702              2.0
2      BGD -205.043430        0  1436.835143      36.258232              3.0
3      NER -173.757336        0   540.306719      25.531997              4.0
4      NPL -169.875497        0   876.400045      46.809570              5.0
5      MRT -162.435554        0  1745.004000      54.266701              6.0
6      NGA -155.768064        0  2233.705969      77.880505              7.0
7      BTN -142.043026        0  3053.174935      97.148056              8.0
8      KEN -142.003447        0  1521.622850      71.896725              9.0
9      UGA -140.276175        0   822.829647      47.845808             10.0
10     BDI -128.980451        0   230.879537      20.137592             11.0
11     LBR -112.690860        0   680.084616      63.502035             12.0

In [36]:
topbot = (cr.sort_values(["cluster","resid_mean"])
            .groupby("cluster", group_keys=False)
            .apply(lambda g: pd.concat([g.head(5), g.tail(5)]))
            .reset_index())  # Country + cluster both as columns, no duplicates
topbot.to_csv("artifacts/cluster_residual_leaderboard.csv", index=False)
print(topbot.head(20))


   Country  resid_mean  cluster    gdp_pc_med  health_pc_med  rank_in_cluster
0      ETH -261.479248        0    659.020879      23.369365              1.0
1      TZA -219.593416        0    974.865570      38.393702              2.0
2      BGD -205.043430        0   1436.835143      36.258232              3.0
3      NER -173.757336        0    540.306719      25.531997              4.0
4      NPL -169.875497        0    876.400045      46.809570              5.0
5      LSO  336.098961        0   1120.337046     117.663960             82.0
6      SWZ  387.616753        0   3932.123261     294.583878             83.0
7      FJI  398.151874        0   4970.976170     177.114021             84.0
8      FSM  408.177000        0   3006.385544     399.253998             85.0
9      KIR  446.171732        0   1761.455794     156.557770             86.0
10     SGP -146.902261        1  57285.195515    2409.339478              1.0
11     ESP -112.520402        1  29468.306035    2696.826050    

C:\Users\sla99\AppData\Local\Temp\ipykernel_9784\624395422.py:3: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



## Visualization

### 0) Load + small helpers

In [38]:
import os, pandas as pd, numpy as np
from pathlib import Path

# 1) Load features (index should be Country)
feats = pd.read_parquet("artifacts/features_country_level.parquet")
if feats.index.name is None:
    feats.index.name = "Country"

# 2) Attach clusters (try saved files; else recompute k=3)
clusters = None
if Path("artifacts/features_country_level_clustered.parquet").exists():
    clusters = pd.read_parquet("artifacts/features_country_level_clustered.parquet")[["cluster"]]
elif Path("artifacts/country_clusters.parquet").exists():
    clusters = pd.read_parquet("artifacts/country_clusters.parquet")[["cluster"]]
elif Path("artifacts/country_clusters.csv").exists():
    tmp = pd.read_csv("artifacts/country_clusters.csv")
    # accept either a 'Country' column or first column as country
    if "Country" in tmp.columns:
        clusters = tmp.set_index("Country")[["cluster"]]
    else:
        clusters = tmp.set_index(tmp.columns[0])[["cluster"]]
else:
    # Recompute quickly (median-impute → scale → KMeans k=3)
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler
    from sklearn.cluster import KMeans
    X = feats.select_dtypes("number").copy()
    mask = (X.isna().mean() <= 0.5) & (X.nunique(dropna=True) > 2)
    X = X.loc[:, mask]
    Xp = SimpleImputer(strategy="median").fit_transform(X)
    Z  = StandardScaler().fit_transform(Xp)
    labels = KMeans(n_clusters=3, n_init=50, random_state=42).fit_predict(Z)
    clusters = pd.DataFrame({"cluster": labels}, index=feats.index)
    clusters.to_parquet("artifacts/features_country_level_clustered.parquet")

# 3) Join and sanity-check
clusters.index.name = feats.index.name
feats = feats.join(clusters, how="left")
missing = feats["cluster"].isna().sum()
print(f"Attached clusters. Countries without a label: {missing}")

# (optional) human-readable names
CLUSTER_NAMES = {
    0: "C0 • Lower income/spend • High mort (improving)",
    1: "C1 • High income/spend • Low mort (improving fast)",
    2: "C2 • Mid income/spend • Mid mort (slower gains)",
}
feats["cluster_name"] = feats["cluster"].map(CLUSTER_NAMES)


Attached clusters. Countries without a label: 0


### 1) World map with legend + counts in title

In [39]:
map_df = feats[["cluster","cluster_name"]].reset_index().rename(columns={"index":"Country"})
counts = map_df["cluster"].value_counts().sort_index().to_dict()
title = "Country clusters (k=3)  —  " + ", ".join([f"C{i}:{counts.get(i,0)}" for i in [0,1,2]])

fig_map = px.choropleth(
    map_df, locations="Country", locationmode="ISO-3",
    color="cluster_name", title=title
)
fig_map.update_layout(legend_title_text="Cluster")
fig_map.write_html("artifacts/map_clusters_k3_legend.html")
display(fig_map)


### 2) Residuals (FE) by cluster — box & violin + top/bottom labels

In [41]:
# Tiny-island flag (for sensitivity notes)
TINY = {"WSM","FSM","KIR","FJI","VUT","SLB","NRU","TON","ATG","GRD","DMA","BRB","STP","MDV","MHL","PLW"}

# Mean residual per country + cluster join
country_resid = (modelA.groupby("Country")[resid_col]
                 .mean()
                 .to_frame("resid_mean")
                 .join(feats[["cluster","cluster_name"]], how="left")
                 .dropna())
country_resid["tiny_island"] = country_resid.index.isin(TINY)

# Box + violin
fig_box = px.box(country_resid, x="cluster_name", y="resid_mean", points="all",
                 color="cluster_name", title="FE residuals by cluster (negative = better)")
fig_violin = px.violin(country_resid, x="cluster_name", y="resid_mean", box=True, points=False,
                       color="cluster_name", title="FE residuals by cluster (violin)")
fig_box.write_html("artifacts/residuals_box_by_cluster.html")
fig_violin.write_html("artifacts/residuals_violin_by_cluster.html")
display(fig_box), display(fig_violin)

# Top/bottom 3 per cluster (CSV)
topbot = (country_resid.sort_values(["cluster","resid_mean"])
          .groupby("cluster", group_keys=False)
          .apply(lambda g: pd.concat([g.head(3), g.tail(3)])))
topbot.to_csv("artifacts/cluster_residual_topbottom3.csv")
topbot.head(12)


C:\Users\sla99\AppData\Local\Temp\ipykernel_9784\2481529250.py:24: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,resid_mean,cluster,cluster_name,tiny_island
Country,,,,
ETH,-261.479248,0,C0 • Lower income/spend • High mort (improving),False
TZA,-219.593416,0,C0 • Lower income/spend • High mort (improving),False
BGD,-205.043430,0,C0 • Lower income/spend • High mort (improving),False
FJI,398.151874,0,C0 • Lower income/spend • High mort (improving),True
FSM,408.177000,0,C0 • Lower income/spend • High mort (improving),True
KIR,446.171732,0,C0 • Lower income/spend • High mort (improving),True
SGP,-146.902261,1,C1 • High income/spend • Low mort (improving f...,False
ESP,-112.520402,1,C1 • High income/spend • Low mort (improving f...,False
ISR,-105.082058,1,C1 • High income/spend • Low mort (improving f...,False


### 3) Within-cluster scatter: Health spend vs Mortality (latest year) with fitted line

In [42]:
# Latest snapshot per country + cluster
latest = panel.groupby("Country")["Year"].max().reset_index()
snap = (latest.merge(panel, on=["Country","Year"], how="left")
              .merge(feats[["cluster","cluster_name"]], left_on="Country", right_index=True, how="left")
              .dropna(subset=["WHS2_131","SH.XPD.CHEX.PC.CD"]))

fig_sc = px.scatter(
    snap, x="SH.XPD.CHEX.PC.CD", y="WHS2_131",
    facet_col="cluster_name", facet_col_wrap=3, trendline="ols",
    log_x=True, hover_name="Country",
    title="Latest year: Health spending (log) vs NCD mortality — by cluster"
)
fig_sc.write_html("artifacts/scatter_spend_vs_mortality_by_cluster.html")
display(fig_sc)

### 4) PCA plot (2D) of feature space, colored by cluster

In [43]:
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

X = feats.select_dtypes(include=[np.number]).drop(columns=["cluster"], errors="ignore")
mask = (X.isna().mean() <= 0.5) & (X.nunique(dropna=True) > 2)
X = X.loc[:, mask]

pipe = Pipeline([("imp", SimpleImputer(strategy="median")),
                 ("sc", StandardScaler())])
Z = pipe.fit_transform(X)

pca = PCA(n_components=2, random_state=42)
Z2 = pca.fit_transform(Z)
pca_df = pd.DataFrame(Z2, index=feats.index, columns=["PC1","PC2"]).join(feats[["cluster_name"]])

fig_pca = px.scatter(pca_df.reset_index(), x="PC1", y="PC2", color="cluster_name",
                     hover_name="Country", title=f"PCA of country features (var exp: PC1={pca.explained_variance_ratio_[0]:.2f}, PC2={pca.explained_variance_ratio_[1]:.2f})")
fig_pca.write_html("artifacts/pca_clusters.html")
display(fig_pca)

### 5) Stability check (k=3 across seeds + ARI vs baseline)

In [44]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

# Reuse Z from PCA section (imputed + scaled features)
base_km = KMeans(n_clusters=3, n_init=50, random_state=42).fit(Z)
base_labels = base_km.labels_
base_sil = silhouette_score(Z, base_labels)

rows = []
for seed in [0,1,2,3,4,10,21,99]:
    km = KMeans(n_clusters=3, n_init=50, random_state=seed).fit(Z)
    labs = km.labels_
    sil = silhouette_score(Z, labs)
    ari = adjusted_rand_score(base_labels, labs)
    rows.append({"seed": seed, "silhouette": sil, "ARI_to_seed42": ari})

stab = pd.DataFrame(rows).sort_values("seed")
stab.to_csv("artifacts/cluster_stability_k3.csv", index=False)
print("Baseline silhouette:", round(base_sil,3))
display(stab)

c:\Users\sla99\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.

c:\Users\sla99\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.

c:\Users\sla99\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.

c:\Users\sla99\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than 

Baseline silhouette: 0.182


,seed,silhouette,ARI_to_seed42
0,0,0.162460,0.333170
1,1,0.162460,0.333170
2,2,0.162460,0.333170
3,3,0.163741,0.350356
4,4,0.162460,0.333170
5,10,0.162460,0.333170
6,21,0.162460,0.333170
7,99,0.181577,1.000000


### 6) Sensitivity note helper (flag tiny islands in leaderboards)

In [45]:
# Add asterisks to tiny islands in the top/bottom table (for slides)
tb_print = topbot.copy()
tb_print["Country_display"] = tb_print.index.where(~tb_print["tiny_island"], tb_print.index + "*")
tb_print[["cluster","Country_display","resid_mean"]].to_csv("artifacts/cluster_residual_topbottom3_annotated.csv")
tb_print.head(12)


,resid_mean,cluster,cluster_name,tiny_island,Country_display
Country,,,,,
ETH,-261.479248,0,C0 • Lower income/spend • High mort (improving),False,ETH
TZA,-219.593416,0,C0 • Lower income/spend • High mort (improving),False,TZA
BGD,-205.043430,0,C0 • Lower income/spend • High mort (improving),False,BGD
FJI,398.151874,0,C0 • Lower income/spend • High mort (improving),True,FJI*
FSM,408.177000,0,C0 • Lower income/spend • High mort (improving),True,FSM*
KIR,446.171732,0,C0 • Lower income/spend • High mort (improving),True,KIR*
SGP,-146.902261,1,C1 • High income/spend • Low mort (improving f...,False,SGP
ESP,-112.520402,1,C1 • High income/spend • Low mort (improving f...,False,ESP
ISR,-105.082058,1,C1 • High income/spend • Low mort (improving f...,False,ISR
